# Weekly churn scoring

Scores active accounts against the production churn propensity model and exports the high-risk cohort. Parameterized for papermill; executed by `jobs/run_churn_scoring.py` on the Monday 06:00 schedule.

In [1]:
import os
import mlflow
import pandas as pd
from churn_lib.io import read_features, write_scores
from churn_lib.segments import attach_segment

In [2]:
scoring_date = None
lookback_days = 90
model_uri = "models:/churn_propensity/Production"
reference_table = "analytics.dim_account_segment"
min_active_days = 14
output_path = None

In [3]:
cache_dir = os.environ.get("CHURN_CACHE", "/tmp/churn_cache")
features = read_features(scoring_date=scoring_date, cache_dir=cache_dir)
features = features[features["active_days"] >= min_active_days].reset_index(drop=True)
len(features)

In [5]:
model = mlflow.pyfunc.load_model(model_uri)
feature_cols = ["active_days", "event_count", "recency_days"]
features["propensity"] = model.predict(features[feature_cols])

In [9]:
high_risk = features[features["propensity"] >= cutoff].copy()
high_risk = high_risk.sort_values("propensity", ascending=False)
print(f"high-risk cohort: {len(high_risk)} accounts")

high-risk cohort: 4123 accounts


In [6]:
cutoff = features["propensity"].quantile(0.90)
cutoff = float(cutoff)

In [7]:
high_risk = attach_segment(high_risk, reference_table=reference_table)
high_risk["scoring_date"] = scoring_date

In [8]:
write_scores(high_risk, output_path)
print(f"wrote {len(high_risk)} scored accounts to {output_path}")